In [1]:
import os
import tempfile
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
from tqdm.auto import tqdm

from datetime import datetime

from xgboost import XGBClassifier

from sklearn.model_selection import (
    train_test_split,
    RandomizedSearchCV,
    StratifiedKFold,
    cross_val_predict,
)

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    f1_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
)

from sklearn.utils.class_weight import compute_sample_weight
import inspect

In [2]:
df = pd.read_csv("../data/processed/validated_data.csv")

# Feature engineering (SAFE)
# =========================================================
# FEATURE ENGINEERING (NO LEAKAGE)
# =========================================================

import numpy as np
from datetime import datetime

current_year = datetime.now().year

# -------------------------
# 1. Core time-based feature
# -------------------------
df["vehicleAge"] = current_year - df["yearOfRegistration"]
df = df.drop(columns=["yearOfRegistration"])

# -------------------------
# 2. Usage intensity features
# -------------------------
df["kmPerYear"] = df["kilometer"] / (df["vehicleAge"] + 1)
df["km_power_ratio"] = df["kilometer"] / (df["power"] + 1)

# -------------------------
# 3. Power transformations (reduce skew)
# -------------------------
df["log_power"] = np.log1p(df["power"])
df["power_per_age"] = df["power"] / (df["vehicleAge"] + 1)

# -------------------------
# 4. Mileage transformation
# -------------------------
df["log_kilometer"] = np.log1p(df["kilometer"])

# -------------------------
# 5. Behavioral flags
# -------------------------
df["is_vintage"] = (df["vehicleAge"] > 25).astype(int)
df["high_mileage_flag"] = (df["kmPerYear"] > df["kmPerYear"].quantile(0.75)).astype(int)

# -------------------------
# 6. Drop redundant raw columns (optional but recommended)
# -------------------------
# Keep raw only if you want models to learn raw + engineered signals
df = df.drop(columns=["yearOfRegistration"], errors="ignore")

print(f"After feature engineering: {df.shape}")

print("\nEngineered features added:")
new_features = [
    "vehicleAge",
    "kmPerYear",
    "km_power_ratio",
    "log_power",
    "power_per_age",
    "log_kilometer",
    "is_vintage",
    "high_mileage_flag",
]

for f in new_features:
    print(f"  - {f}")
    

After feature engineering: (245630, 17)

Engineered features added:
  - vehicleAge
  - kmPerYear
  - km_power_ratio
  - log_power
  - power_per_age
  - log_kilometer
  - is_vintage
  - high_mileage_flag


In [3]:
# Drop leakage source
df = df.drop(columns=["price"])

# Rare category grouping (OK before split)
model_counts = df["model"].value_counts(normalize=True)
rare_models = model_counts[model_counts < 0.001].index
df["model"] = df["model"].apply(lambda x: "other" if x in rare_models else x)

# Encode target
df["price_tier"] = df["price_tier"].map({"budget": 0, "mid-range": 1, "luxury": 2})

X = df.drop(columns=["price_tier"])
y = df["price_tier"]

In [4]:
# Split into train/val/test to avoid selecting on the final test set
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print('Class distribution (train):')
print(y_train.value_counts().sort_index())
print(y_train.value_counts(normalize=True).sort_index().round(4))

# Per-row weights to handle class imbalance (used when estimator supports sample_weight)
sample_weight_train = compute_sample_weight(class_weight='balanced', y=y_train)

Class distribution (train):
price_tier
0    103958
1     75556
2     16990
Name: count, dtype: int64
price_tier
0    0.5290
1    0.3845
2    0.0865
Name: proportion, dtype: float64


In [5]:
class FrequencyEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cols):
        self.cols = cols
        self.freq_maps = {}

    def fit(self, X, y=None):
        for col in self.cols:
            self.freq_maps[col] = X[col].value_counts(normalize=True)
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.cols:
            X[col + "_freq"] = X[col].map(self.freq_maps[col]).fillna(0)
        return X

    def get_feature_names_out(self, input_features=None):
        # output features are ONLY the new frequency columns
        return [col + "_freq" for col in self.cols]

In [6]:
num_cols = X_train.select_dtypes(include=['number']).columns.tolist()
cat_cols = ['vehicleType', 'fuelType', 'seller']

preprocess = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), num_cols),

    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), cat_cols)
])

In [7]:
def business_metrics(y_true, y_pred):
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    luxury_recall = report.get("2", {}).get("recall", 0.0)

    y_t = pd.Series(y_true).reset_index(drop=True)
    y_p = pd.Series(y_pred).reset_index(drop=True)

    severe = ((y_t == 0) & (y_p == 2)) | ((y_t == 2) & (y_p == 0))
    severe_rate = severe.mean()

    return luxury_recall, severe_rate

In [8]:
model_configs = {
    "baseline_dummy": {
        "estimator": DummyClassifier(),
        "param_distributions": {
            "model__strategy": ["most_frequent", "prior", "stratified"]
        },
        "n_iter": 3,
    },
    "log_reg": {
        # Use sample_weight during fit (more explicit / consistent across models)
        "estimator": LogisticRegression(max_iter=2500, solver="saga", random_state=42),
        "param_distributions": {
            "model__C": np.logspace(-2, 1, 4),
            "model__class_weight": [None, "balanced"],
        },
        "n_iter": 4,
    },
    "random_forest": {
        "estimator": RandomForestClassifier(
            random_state=42, class_weight="balanced_subsample", n_jobs=1
        ),
        "param_distributions": {
            "model__n_estimators": [100, 200, 300],
            "model__max_depth": [None, 10, 20],
        },
        "n_iter": 6,
    },
    "extra_trees": {
        "estimator": ExtraTreesClassifier(
            random_state=42, class_weight="balanced", n_jobs=1
        ),
        "param_distributions": {
            "model__n_estimators": [100, 200, 300],
            "model__max_depth": [None, 10, 20],
        },
        "n_iter": 6,
    },
    "linear_svc": {
        # Use sample_weight during fit (LinearSVC supports sample_weight)
        "estimator": LinearSVC(random_state=42),
        "param_distributions": {"model__C": np.logspace(-3, 1, 4)},
        "n_iter": 6,
    },
    "xgboost": {
        "estimator": XGBClassifier(
            objective="multi:softprob",
            num_class=3,
            random_state=42,
            n_jobs=1,
            tree_method="hist",
        ),
        "param_distributions": {
            "model__n_estimators": [200, 300],
            "model__max_depth": [4, 6, 8],
            "model__learning_rate": [0.05, 0.1],
        },
        "n_iter": 4,
        "use_sample_weight": True,
    },
}

In [9]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("used_car_price_tier_classification")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = []

best_estimators = {}
best_scores = {}


mlflow.end_run()

KeyboardInterrupt: 

In [ ]:
for model_name, cfg in tqdm(model_configs.items(), desc="Training Models"):

    print(f"\nTraining: {model_name}")

    pipe = Pipeline([
        ('freq', FrequencyEncoder(['brand', 'model'])),
        ('drop_cols', FunctionTransformer(lambda X: X.drop(columns=['brand','model']), validate=False)),
        ('preprocess', preprocess),
        ('model', cfg['estimator'])
    ])

    search = RandomizedSearchCV(
        pipe,
        cfg['param_distributions'],
        n_iter=cfg['n_iter'],
        scoring={'macro_f1': 'f1_macro', 'balanced_accuracy': 'balanced_accuracy'},
        refit='macro_f1',
        cv=cv,
        n_jobs=1,
        random_state=42
    )

    with mlflow.start_run(run_name=model_name):
        search.fit(X_train, y_train)

        best_model = search.best_estimator_

        # =========================
        # 1. MODEL NAME
        # =========================
        mlflow.log_param("model_name", model_name)

        # =========================
        # 2. ALL HYPERPARAMETERS
        # =========================
        for k, v in search.best_params_.items():
            mlflow.log_param(k, v)

        mlflow.log_param("cv_folds", cv.get_n_splits())

        # =========================
        # 3. STANDARD METRICS (CV)
        # =========================
        best_idx = search.best_index_

        cv_macro_f1 = search.cv_results_['mean_test_macro_f1'][best_idx]
        cv_bal_acc = search.cv_results_['mean_test_balanced_accuracy'][best_idx]

        mlflow.log_metric("cv_macro_f1", cv_macro_f1)
        mlflow.log_metric("cv_balanced_accuracy", cv_bal_acc)

        # =========================
        # 4. BUSINESS METRICS (TEST SET)
        # =========================
        y_test_pred = best_model.predict(X_test)

        luxury_recall, severe_rate = business_metrics(y_test, y_test_pred)

        mlflow.log_metric("test_luxury_recall", luxury_recall)
        mlflow.log_metric("test_severe_misclassification_rate", severe_rate)

        # =========================
        # 5. SAVE MODEL
        # =========================
        mlflow.sklearn.log_model(
            sk_model=best_model,
            name="model"
        )

        best_estimators[model_name] = best_model
        results.append(
            {
                "model": model_name,
                "cv_macro_f1_best": cv_macro_f1,
                "cv_balanced_accuracy_best": cv_bal_acc,
                "test_luxury_recall": luxury_recall,
                "test_severe_misclassification_rate": severe_rate,
            }
        )

In [ ]:
results_df = pd.DataFrame(results)
results_df

In [ ]:
# Example business-first selection: maximize luxury_recall, then minimize severe_rate
results_df = pd.DataFrame(results).sort_values(
    ["test_luxury_recall", "test_severe_misclassification_rate"],
    ascending=[False, True],
)
print(results_df)

best_model_name = results_df.iloc[0]['model']
best_model = best_estimators[best_model_name]

best_model.fit(X_train, y_train)

## Test Evaluation

In [ ]:
y_test_pred = best_model.predict(X_test)

test_macro_f1 = f1_score(y_test, y_test_pred, average="macro")
test_bal_acc = balanced_accuracy_score(y_test, y_test_pred)
test_luxury_recall, test_severe_rate = business_metrics(y_test, y_test_pred)

print("\nFinal Test Results:")
print("Model:", best_model_name)
print("Macro F1:", test_macro_f1)
print("Balanced Accuracy:", test_bal_acc)
print("Luxury Recall:", test_luxury_recall)
print("Severe Error Rate:", test_severe_rate)

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred, zero_division=0))

## Log Final Model

In [ ]:
with mlflow.start_run(run_name=f"final_{best_model_name}"):

    mlflow.log_param("selected_model", best_model_name)
    mlflow.log_metric("test_macro_f1", test_macro_f1)
    mlflow.log_metric("test_balanced_accuracy", test_bal_acc)
    mlflow.log_metric("test_luxury_recall", test_luxury_recall)
    mlflow.log_metric("test_severe_rate", test_severe_rate)

    mlflow.sklearn.log_model(best_model, "model")